# ProAg Track 2 — Producer Analytics Pipeline

End-to-end notebook for the 7 Track 2 producer files. Run cells top to bottom.

**Stages**
1. Setup & config
2. Helpers (data quality + site mapping)
3. Ingest + clean each of the 7 files
4. Stitch cycles
5. Allocate costs (confidence-scored)
6. Hedge P&L
7. Per-cycle P&L
8. Anomaly flags
9. Save outputs + headline summary

In [25]:
pip install pandas numpy jupyter


[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## 1. Setup & config

In [26]:
from pathlib import Path
from dataclasses import dataclass, field
import numpy as np
import pandas as pd

# ---- EDIT THIS to point at your Track 2 folder ----
DATA_DIR = Path("/workspaces/CCDS/Data")
OUTPUT_DIR = Path("/workspaces/CCDS/Outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PRODUCER_FILES = {
    "sow_farm":    "2025_dummy_sow_farm_weekly_farrowing.csv",
    "nursery":     "2025_dummy_nursery_intake.csv",
    "pig_flow":    "2025_dummy_barn_to_barn_pig_flow.csv",
    "environment": "2025_dummy_barn_environmental_utilities.csv",
    "packer":      "2025_dummy_packer_settlement.csv",
    "hedging":     "2025_dummy_hog_hedging_aligned_to_nursery.csv",
    "accounting":  "2025_swine_accounting_dummy .csv",   # trailing space
}

SITE_CANONICAL_MAP = {
    "Site A": "Site A", "Site B": "Site B", "Site C": "Site C",
    "Nursery North": "Nursery North", "Nursery South": "Nursery South",
    "Summer Creek": "Summer Creek", "Sow Farm Alpha": "Sow Farm Alpha",
    "Demo Producer A": "Demo Producer A",
    "Finisher East": "Finisher East", "Finisher West": "Finisher West",
    "External Source": "External Source",
}

Z_THRESHOLD = 2.5
ASSUMED_CARC_WT_LB = 215.0

print(f"Data dir: {DATA_DIR.resolve()}")
print(f"Output dir: {OUTPUT_DIR.resolve()}")
assert DATA_DIR.exists(), f"Data dir not found: {DATA_DIR}"
for f in PRODUCER_FILES.values():
    assert (DATA_DIR / f).exists(), f"Missing file: {f}"
print("All 7 files present.")

Data dir: /workspaces/CCDS/Data
Output dir: /workspaces/CCDS/Outputs
All 7 files present.


## 2. Data-quality helpers

Reusable functions used by every cleaning cell below. Each one **records what it does** in a `TableReport` so you can audit it later.

In [27]:
@dataclass
class TableReport:
    table: str
    rows_in: int = 0
    rows_out: int = 0
    nulls: dict = field(default_factory=dict)
    issues: list = field(default_factory=list)

    def add_issue(self, kind, column, count, action, note=""):
        self.issues.append({
            "table": self.table, "kind": kind, "column": column,
            "count": count, "action": action, "note": note,
        })

    def summary(self):
        dropped = self.rows_in - self.rows_out
        line = f"{self.table}: {self.rows_in} -> {self.rows_out}"
        if dropped:
            line += f"  (dropped {dropped})"
        return line

def _null_counts(df):
    return {c: int(df[c].isna().sum()) for c in df.columns if df[c].isna().any()}

def _drop_required_nulls(df, required, r):
    mask = df[required].isna().any(axis=1)
    n = int(mask.sum())
    if n:
        r.add_issue("required_null", ",".join(required), n, "dropped_row")
    return df.loc[~mask].copy()

def _drop_duplicates(df, keys, r):
    before = len(df)
    out = df.drop_duplicates(subset=keys, keep="first")
    dropped = before - len(out)
    if dropped:
        r.add_issue("duplicate", ",".join(keys), dropped, "dropped_row")
    return out

def _coerce_numeric(df, cols, r):
    for c in cols:
        if c not in df.columns:
            continue
        before = df[c].isna().sum()
        df[c] = pd.to_numeric(df[c], errors="coerce")
        new = df[c].isna().sum() - before
        if new > 0:
            r.add_issue("non_numeric", c, int(new), "coerced_to_nan")
    return df

def _flag_out_of_range(df, col, low, high, r, action="set_null"):
    if col not in df.columns:
        return df
    mask = pd.Series(False, index=df.index)
    if low is not None:
        mask |= df[col] < low
    if high is not None:
        mask |= df[col] > high
    n = int(mask.sum())
    if n:
        r.add_issue("out_of_range", col, n, action, note=f"Outside [{low}, {high}]")
        if action == "set_null":
            df.loc[mask, col] = np.nan
        elif action == "dropped_row":
            df = df.loc[~mask].copy()
    return df

def _canonical_site(val):
    if pd.isna(val):
        return val
    return SITE_CANONICAL_MAP.get(str(val).strip(), str(val).strip())

all_reports = []   # collected as we run each cleaning cell
print("Helpers defined.")

Helpers defined.


## 3. Ingest + clean — one cell per file

### 3.1 Sow farm

In [28]:
sow_farm = pd.read_csv(DATA_DIR / PRODUCER_FILES["sow_farm"])
sow_farm["Week_Start"] = pd.to_datetime(sow_farm["Week_Start"], errors="coerce")

r = TableReport("sow_farm", rows_in=len(sow_farm))
r.nulls = _null_counts(sow_farm)
sow_farm = _coerce_numeric(sow_farm, [
    "Sows_Bred", "Farrowings", "Total_Born", "Live_Born",
    "PreWean_Mortality_Head", "Weaned_Pigs",
    "Avg_Wean_Weight_lb", "Avg_Wean_Age_Days",
], r)
sow_farm = _drop_required_nulls(sow_farm, ["Week_Start", "Sow_Farm"], r)
sow_farm = _flag_out_of_range(sow_farm, "Avg_Wean_Weight_lb", 5, 30, r)
sow_farm = _flag_out_of_range(sow_farm, "Avg_Wean_Age_Days", 10, 35, r)
r.rows_out = len(sow_farm)
all_reports.append(r)

print(r.summary())
sow_farm.head()

sow_farm: 156 -> 156


,Week_Start,Sow_Farm,Sows_Bred,Farrowings,Total_Born,Live_Born,PreWean_Mortality_Head,Weaned_Pigs,Avg_Wean_Weight_lb,Avg_Wean_Age_Days,Notes
0,2025-01-06,Sow Farm Alpha,553,478,6473,5881,514,5367,13.58,19,Synthetic sow farm production data
1,2025-01-13,Sow Farm Alpha,628,551,7595,7045,521,6524,11.95,21,Synthetic sow farm production data
2,2025-01-20,Sow Farm Alpha,643,539,7344,6831,609,6222,12.19,20,Synthetic sow farm production data
3,2025-01-27,Sow Farm Alpha,553,455,6236,5670,565,5105,12.80,20,Synthetic sow farm production data
4,2025-02-03,Sow Farm Alpha,596,514,7643,7187,547,6640,12.39,21,Synthetic sow farm production data


### 3.2 Nursery intake

In [29]:
nursery = pd.read_csv(DATA_DIR / PRODUCER_FILES["nursery"])
nursery["Placement_Date"] = pd.to_datetime(nursery["Placement_Date"], errors="coerce")
nursery["Nursery_Site"] = nursery["Nursery_Site"].map(_canonical_site)

r = TableReport("nursery", rows_in=len(nursery))
r.nulls = _null_counts(nursery)
nursery = _coerce_numeric(nursery, [
    "Expected_Head", "Received_Head", "Avg_Weight_In_lb", "Age_Days",
], r)
nursery = _drop_required_nulls(
    nursery, ["Pig_Group_ID", "Placement_Date", "Received_Head"], r)
nursery = _drop_duplicates(nursery, ["Pig_Group_ID"], r)
nursery = _flag_out_of_range(nursery, "Received_Head", 100, 5000, r, "dropped_row")
nursery = _flag_out_of_range(nursery, "Avg_Weight_In_lb", 5, 30, r)
nursery = _flag_out_of_range(nursery, "Age_Days", 10, 60, r)
r.rows_out = len(nursery)
all_reports.append(r)

print(r.summary())
nursery.head()

nursery: 18 -> 18


,Pig_Group_ID,Placement_Date,Nursery_Site,Nursery_Barn,Pig_Source,Production_Flow,Expected_Head,Received_Head,Avg_Weight_In_lb,Age_Days,Health_Status,Placement_Week,Notes
0,PG-1017,2025-01-16,Nursery North,N1,External Wean Pig Supplier,Wean-to-Finish,2338,2285,11.55,21,Monitored,3,Dummy nursery intake data for capacity and flo...
1,PG-1002,2025-01-24,Nursery South,NS2,External Wean Pig Supplier,Wean-to-Finish,2281,2274,12.49,21,Negative,4,Dummy nursery intake data for capacity and flo...
2,PG-1014,2025-02-02,Nursery South,NS2,Internal Sow Farm,Wean-to-Finish,2454,2431,13.97,20,Negative,5,Dummy nursery intake data for capacity and flo...
3,PG-1011,2025-02-11,Nursery South,NS2,Internal Sow Farm,Wean-to-Finish,2383,2350,12.87,21,Negative,7,Dummy nursery intake data for capacity and flo...
4,PG-1015,2025-02-15,Nursery South,NS2,Internal Sow Farm,Wean-to-Finish,2596,2569,12.52,20,Negative,7,Dummy nursery intake data for capacity and flo...


### 3.3 Pig flow

In [30]:
pig_flow = pd.read_csv(DATA_DIR / PRODUCER_FILES["pig_flow"])
pig_flow["Movement_Date"] = pd.to_datetime(pig_flow["Movement_Date"], errors="coerce")
pig_flow["From_Site"] = pig_flow["From_Site"].map(_canonical_site)
pig_flow["To_Site"] = pig_flow["To_Site"].map(_canonical_site)

r = TableReport("pig_flow", rows_in=len(pig_flow))
r.nulls = _null_counts(pig_flow)
pig_flow = _coerce_numeric(pig_flow, ["Head_Moved", "Avg_Weight_lb"], r)
pig_flow = _drop_required_nulls(
    pig_flow, ["Pig_Group_ID", "Movement_Date", "Head_Moved"], r)
pig_flow = _flag_out_of_range(pig_flow, "Head_Moved", 1, 5000, r, "dropped_row")
pig_flow = _flag_out_of_range(pig_flow, "Avg_Weight_lb", 5, 350, r)
if pig_flow["From_Site"].isna().any():
    n = int(pig_flow["From_Site"].isna().sum())
    pig_flow["From_Site"] = pig_flow["From_Site"].fillna("External Source")
    r.add_issue("optional_null", "From_Site", n, "filled_external_source")
r.rows_out = len(pig_flow)
all_reports.append(r)

print(r.summary())
pig_flow.head()

pig_flow: 41 -> 41


,Pig_Group_ID,Movement_Date,From_Site,From_Barn,To_Site,To_Barn,Event,Head_Moved,Avg_Weight_lb,Notes
0,PG-1000,2025-03-01,External Source,NaN,Nursery North,N1,Placement,2248,12.45,Nursery placement
1,PG-1000,2025-04-24,Nursery North,N1,Finisher East,F1,Transfer,2150,52.01,Nursery to finisher
2,PG-1001,2025-06-13,External Source,NaN,Nursery South,NS1,Placement,2570,10.43,Nursery placement
3,PG-1001,2025-08-11,Nursery South,NS1,Finisher East,F3,Transfer,2524,63.38,Nursery to finisher
4,PG-1002,2025-01-24,External Source,NaN,Nursery South,NS2,Placement,2274,12.49,Nursery placement


### 3.4 Environment / utilities

In [31]:
environment = pd.read_csv(DATA_DIR / PRODUCER_FILES["environment"])
environment["Date"] = pd.to_datetime(environment["Date"], errors="coerce")
environment["Site"] = environment["Site"].map(_canonical_site)

r = TableReport("environment", rows_in=len(environment))
r.nulls = _null_counts(environment)
numeric_cols = [
    "Avg_Temperature_F", "Avg_Humidity_pct",
    "Water_Usage_Gallons", "Power_Usage_kWh", "Ventilation_Level_pct",
]
environment = _coerce_numeric(environment, numeric_cols, r)
environment = _drop_required_nulls(environment, ["Date", "Site", "Barn"], r)
environment = _flag_out_of_range(environment, "Avg_Temperature_F", 40, 110, r)
environment = _flag_out_of_range(environment, "Avg_Humidity_pct", 0, 100, r)
environment = _flag_out_of_range(environment, "Water_Usage_Gallons", 0, 50000, r)
environment = _flag_out_of_range(environment, "Power_Usage_kWh", 0, 10000, r)
environment = _flag_out_of_range(environment, "Ventilation_Level_pct", 0, 100, r)

# Forward-fill short sensor gaps within (Site, Barn)
environment = environment.sort_values(["Site", "Barn", "Date"])
ffilled = 0
for col in ["Avg_Temperature_F", "Avg_Humidity_pct", "Ventilation_Level_pct"]:
    before = environment[col].isna().sum()
    environment[col] = environment.groupby(["Site", "Barn"])[col].transform(
        lambda s: s.ffill(limit=2)
    )
    ffilled += int(before - environment[col].isna().sum())
if ffilled:
    r.add_issue("imputed", "sensor_columns", ffilled, "forward_filled",
                note="Forward-filled up to 2 days within (Site, Barn)")
r.rows_out = len(environment)
all_reports.append(r)

print(r.summary())
for issue in r.issues:
    print(f"  - {issue['kind']}: {issue['column']} n={issue['count']} ({issue['action']})")
environment.head()

environment: 3285 -> 3285
  - out_of_range: Avg_Temperature_F n=122 (set_null)
  - imputed: sensor_columns n=96 (forward_filled)


,Date,Site,Barn,Avg_Temperature_F,Avg_Humidity_pct,Water_Usage_Gallons,Power_Usage_kWh,Ventilation_Level_pct,Notes
1460,2025-01-01,Finisher East,F1,60.4,62.5,5457,559.7,82.9,Synthetic environmental + utility data for pla...
1461,2025-01-02,Finisher East,F1,60.1,67.5,5330,538.1,60.4,Synthetic environmental + utility data for pla...
1462,2025-01-03,Finisher East,F1,58.9,59.3,5606,518.4,61.7,Synthetic environmental + utility data for pla...
1463,2025-01-04,Finisher East,F1,62.3,74.7,5401,509.3,41.5,Synthetic environmental + utility data for pla...
1464,2025-01-05,Finisher East,F1,59.6,63.7,4492,572.3,50.9,Synthetic environmental + utility data for pla...


### 3.5 Packer settlement

In [32]:
packer = pd.read_csv(DATA_DIR / PRODUCER_FILES["packer"])
for c in ("Kill_Date", "Delivery_Date", "Payment_Date"):
    packer[c] = pd.to_datetime(packer[c], errors="coerce")
packer["Site"] = packer["Site"].map(_canonical_site)

r = TableReport("packer", rows_in=len(packer))
r.nulls = _null_counts(packer)
packer = _coerce_numeric(packer, [
    "Net_Payment_$", "Gross_Payment_$", "Eval_Price_CWT",
    "Avg_Carc_Wt_lb", "Avg_Live_Wt_lb", "Avg_Yield_pct",
    "Delivered_Head", "Paid_Head",
], r)
packer = _drop_required_nulls(
    packer, ["Settlement_ID", "Kill_Date", "Site", "Net_Payment_$"], r)
packer = _drop_duplicates(packer, ["Settlement_ID"], r)
packer = _flag_out_of_range(packer, "Avg_Yield_pct", 50, 90, r)
packer = _flag_out_of_range(packer, "Avg_Carc_Wt_lb", 100, 350, r)
packer = _flag_out_of_range(packer, "Delivered_Head", 1, 500, r, "dropped_row")
r.rows_out = len(packer)
all_reports.append(r)

print(r.summary())
packer.head()

packer: 180 -> 180


,Packer,Plant,Settlement_ID,Scale_Ticket_ID,Kill_Date,Delivery_Date,Producer,Site,Tattoo,Load_ID,...,Gross_Payment_$,Freight_Carrier,Freight_Amount_$,Checkoff_$,Yardage_$,Other_Deductions_$,Total_Deductions_$,Net_Payment_$,Payment_Date,Comments
0,JBS,"Sioux Falls, SD",SET-20250103-1455720,1455720,2025-01-03,2025-01-02,Demo Producer C,Summer Creek,4574,LD-19458,...,37473.36,BUSMA TRUCKING,625,70.8,26.55,24.60,746.95,36726.41,2025-01-13,Dummy packer settlement data for student testing
1,JBS,"Storm Lake, IA",SET-20250103-1554286,1554286,2025-01-03,2025-01-03,Demo Producer B,Summer Creek,1505,LD-86318,...,32573.66,Dakota Hauling,650,64.8,24.30,10.87,749.97,31823.69,2025-01-08,Dummy packer settlement data for student testing
2,JBS,"Worthington, MN",SET-20250103-1711205,1711205,2025-01-03,2025-01-02,Demo Producer B,Summer Creek,8509,LD-25376,...,29302.15,Dakota Hauling,625,72.8,27.30,62.30,787.40,28514.75,2025-01-12,Dummy packer settlement data for student testing
3,Seaboard,"Sioux Falls, SD",SET-20250103-1932501,1932501,2025-01-03,2025-01-02,Demo Producer A,Warbler Site,6983,LD-47163,...,35110.33,Tri-State Livestock,675,65.6,24.60,90.54,855.74,34254.59,2025-01-13,Dummy packer settlement data for student testing
4,Tyson (IBP),"Worthington, MN",SET-20250103-1232317,1232317,2025-01-03,2025-01-02,Demo Producer A,Warbler Site,9217,LD-76573,...,28216.36,Prairie Transport,700,68.4,25.65,89.57,883.62,27332.74,2025-01-10,Dummy packer settlement data for student testing


### 3.6 Hedging

In [33]:
hedging = pd.read_csv(DATA_DIR / PRODUCER_FILES["hedging"])
for c in ("Trade_Date", "Expiration_Date"):
    hedging[c] = pd.to_datetime(hedging[c], errors="coerce")

r = TableReport("hedging", rows_in=len(hedging))
r.nulls = _null_counts(hedging)
hedging = _coerce_numeric(hedging, [
    "Expected_Head_From_Nursery", "Coverage_Percent", "Head_Covered",
    "Contracts", "Strike_or_Futures_Price_CWT", "Option_or_LRP_Premium_CWT",
], r)
hedging = _drop_required_nulls(hedging, [
    "Pig_Group_ID", "Trade_Date", "Expiration_Date",
    "Strike_or_Futures_Price_CWT", "Head_Covered",
], r)
hedging = _flag_out_of_range(hedging, "Coverage_Percent", 0, 1.5, r)
hedging = _flag_out_of_range(hedging, "Strike_or_Futures_Price_CWT", 30, 200, r)
hedging = _flag_out_of_range(hedging, "Head_Covered", 1, 10000, r, "dropped_row")
r.rows_out = len(hedging)
all_reports.append(r)

print(r.summary())
hedging.head()

hedging: 18 -> 18


,Pig_Group_ID,Expected_Head_From_Nursery,Coverage_Percent,Head_Covered,Contracts,Instrument_Type,Buy_Sell,Trade_Date,Expiration_Date,Strike_or_Futures_Price_CWT,Option_or_LRP_Premium_CWT,Contract_Month,Broker,Notes
0,PG-1017,2338,0.86,2000,5,Lean Hog Futures,Sell,2025-02-08,2025-06-10,79.93,0.00,25-Jun,ProAg Risk,Coverage sized off nursery expected head
1,PG-1014,2454,0.65,1600,4,Lean Hog Futures,Sell,2025-02-23,2025-06-07,93.15,0.00,25-Jun,ProAg Risk,Coverage sized off nursery expected head
2,PG-1002,2281,0.53,1200,3,Lean Hog Put Option,Buy,2025-03-07,2025-06-30,88.20,4.09,25-Jun,ProAg Risk,Coverage sized off nursery expected head
3,PG-1015,2596,0.62,1600,4,LRP Policy,Buy,2025-03-17,2025-07-07,79.17,3.30,25-Jul,ProAg Risk,Coverage sized off nursery expected head
4,PG-1000,2305,0.52,1200,3,Lean Hog Put Option,Buy,2025-03-30,2025-08-23,89.89,4.18,25-Aug,ProAg Risk,Coverage sized off nursery expected head


### 3.7 Accounting

In [34]:
accounting = pd.read_csv(DATA_DIR / PRODUCER_FILES["accounting"])
accounting["Date"] = pd.to_datetime(accounting["Date"], errors="coerce")
accounting["Site"] = accounting["Site"].map(_canonical_site)

r = TableReport("accounting", rows_in=len(accounting))
r.nulls = _null_counts(accounting)
accounting = _coerce_numeric(accounting, ["Quantity", "Unit_Cost", "Total_Cost"], r)
accounting = _drop_required_nulls(
    accounting, ["Date", "Site", "Cost_Category", "Total_Cost"], r)
accounting = _flag_out_of_range(accounting, "Total_Cost", 0, 1_000_000, r)
if accounting["Vendor"].isna().any():
    n = int(accounting["Vendor"].isna().sum())
    accounting["Vendor"] = accounting["Vendor"].fillna("Unknown Vendor")
    r.add_issue("optional_null", "Vendor", n, "filled_unknown")
r.rows_out = len(accounting)
all_reports.append(r)

print(r.summary())
accounting.head()

accounting: 360 -> 360


,Date,Fiscal_Year,Month,Operation_ID,Site,Cost_Category,Cost_Subcategory,Description,Quantity,Unit,Unit_Cost,Total_Cost,Vendor,Production_Phase,Pig_Group_ID,Notes
0,2025-04-13,2025,4,OP-2025-001,Site A,Facilities,Utilities,Utilities expense,92.53,unit,35.20,3257.06,VetHealth Inc,Finishing,PG-5426,Dummy data for student testing
1,2025-03-29,2025,3,OP-2025-001,Site B,Labor,Payroll Burden,Payroll Burden expense,11.27,unit,43.66,492.05,VetHealth Inc,Gilt Development,PG-6051,Dummy data for student testing
2,2025-10-04,2025,10,OP-2025-001,Site A,Feed,Feed Delivery,Feed Delivery expense,306.21,unit,0.81,248.03,AgriFeed Co,Finishing,PG-3558,Dummy data for student testing
3,2025-06-19,2025,6,OP-2025-001,Site C,Facilities,Barn Supplies,Barn Supplies expense,183.81,unit,20.80,3823.25,Midwest Transport,Gilt Development,PG-8734,Dummy data for student testing
4,2025-09-01,2025,9,OP-2025-001,Site A,Feed,Finisher Feed,Finisher Feed expense,430.11,unit,30.77,13234.48,VetHealth Inc,Gilt Development,PG-9792,Dummy data for student testing


### 3.8 Data-quality summary across all 7 files

In [35]:
print("=" * 70)
print(" DATA QUALITY SUMMARY")
print("=" * 70)
for r in all_reports:
    print("  " + r.summary())
    for issue in r.issues:
        if issue["action"] in ("dropped_row", "set_null", "coerced_to_nan"):
            print(f"      ! {issue['kind']:<14} {str(issue['column'])[:24]:<24} "
                  f"n={issue['count']:<5} {issue['action']}")

# Build the long-format DQ report dataframe
dq_rows = []
for r in all_reports:
    for col, n in r.nulls.items():
        dq_rows.append({"table": r.table, "kind": "null_count", "column": col,
                        "count": n, "action": "reported",
                        "note": f"{n}/{r.rows_in} ({n/max(r.rows_in,1):.1%})"})
    dq_rows.extend(r.issues)
    dq_rows.append({"table": r.table, "kind": "row_count", "column": None,
                    "count": r.rows_out, "action": "kept",
                    "note": f"in={r.rows_in} out={r.rows_out}"})
dq_report = pd.DataFrame(dq_rows)
dq_report.head(15)

 DATA QUALITY SUMMARY
  sow_farm: 156 -> 156
  nursery: 18 -> 18
  pig_flow: 41 -> 41
  environment: 3285 -> 3285
      ! out_of_range   Avg_Temperature_F        n=122   set_null
  packer: 180 -> 180
  hedging: 18 -> 18
  accounting: 360 -> 360


,table,kind,column,count,action,note
0,sow_farm,row_count,NaN,156,kept,in=156 out=156
1,nursery,row_count,NaN,18,kept,in=18 out=18
2,pig_flow,null_count,From_Barn,18,reported,18/41 (43.9%)
3,pig_flow,row_count,NaN,41,kept,in=41 out=41
4,environment,out_of_range,Avg_Temperature_F,122,set_null,"Outside [40, 110]"
5,environment,imputed,sensor_columns,96,forward_filled,"Forward-filled up to 2 days within (Site, Barn)"
6,environment,row_count,NaN,3285,kept,in=3285 out=3285
7,packer,row_count,NaN,180,kept,in=180 out=180
8,hedging,row_count,NaN,18,kept,in=18 out=18
9,accounting,row_count,NaN,360,kept,in=360 out=360


## 4. Stitch cycles

One row per `Pig_Group_ID` with placement and last-movement summary.

In [36]:
base = nursery[[
    "Pig_Group_ID", "Placement_Date", "Nursery_Site", "Nursery_Barn",
    "Production_Flow", "Expected_Head", "Received_Head", "Avg_Weight_In_lb",
]].rename(columns={
    "Pig_Group_ID": "cycle_id",
    "Placement_Date": "placement_date",
    "Nursery_Site": "nursery_site",
    "Nursery_Barn": "nursery_barn",
    "Production_Flow": "production_flow",
    "Expected_Head": "expected_head",
    "Received_Head": "received_head",
    "Avg_Weight_In_lb": "avg_weight_in_lb",
})

last_move = (
    pig_flow.groupby("Pig_Group_ID")
    .agg(last_move_date=("Movement_Date", "max"),
         movement_count=("Movement_Date", "count"))
    .reset_index()
    .rename(columns={"Pig_Group_ID": "cycle_id"})
)

cycles = base.merge(last_move, on="cycle_id", how="left")
print(f"Built {len(cycles)} cycles.")
cycles

Built 18 cycles.


,cycle_id,placement_date,nursery_site,nursery_barn,production_flow,expected_head,received_head,avg_weight_in_lb,last_move_date,movement_count
0,PG-1017,2025-01-16,Nursery North,N1,Wean-to-Finish,2338,2285,11.55,2025-03-25,3
1,PG-1002,2025-01-24,Nursery South,NS2,Wean-to-Finish,2281,2274,12.49,2025-03-23,2
2,PG-1014,2025-02-02,Nursery South,NS2,Wean-to-Finish,2454,2431,13.97,2025-04-11,2
3,PG-1011,2025-02-11,Nursery South,NS2,Wean-to-Finish,2383,2350,12.87,2025-04-20,2
4,PG-1015,2025-02-15,Nursery South,NS2,Wean-to-Finish,2596,2569,12.52,2025-04-17,2
5,PG-1016,2025-02-22,Nursery South,NS2,Wean-to-Finish,2406,2399,13.43,2025-05-02,2
6,PG-1000,2025-03-01,Nursery North,N1,Nursery-to-Finish,2305,2248,12.45,2025-04-24,2
7,PG-1004,2025-03-21,Nursery South,NS1,Wean-to-Finish,2139,2106,13.61,2025-05-23,3
8,PG-1005,2025-03-23,Nursery North,N2,Wean-to-Finish,2115,2107,11.90,2025-05-17,2
9,PG-1009,2025-04-06,Nursery North,N1,Wean-to-Finish,2512,2498,11.67,2025-06-09,3


## 5. Allocate costs (confidence-scored)

Maps accounting rows to cycles with HIGH / MEDIUM / LOW confidence:
- **HIGH** — exact `Pig_Group_ID` match
- **MEDIUM** — Site + active-cycle window match
- **LOW** — Site only fallback (split equally)

In [37]:
valid_ids = set(cycles["cycle_id"])
site_to_cycles = {site: g for site, g in cycles.groupby("nursery_site")}

rows = []
for _, acct in accounting.iterrows():
    cid = acct.get("Pig_Group_ID")
    date = acct["Date"]
    cost = acct["Total_Cost"]
    site = acct["Site"]

    # HIGH — exact Pig_Group_ID match
    if pd.notna(cid) and cid in valid_ids:
        rows.append({
            "cycle_id": cid, "date": date,
            "cost_category": acct["Cost_Category"],
            "cost_subcategory": acct["Cost_Subcategory"],
            "allocated_cost": cost, "confidence": "HIGH",
            "vendor": acct["Vendor"],
        })
        continue

    site_matches = site_to_cycles.get(site)
    if site_matches is None or site_matches.empty:
        continue

    # MEDIUM — Site + active-cycle window
    active = site_matches[
        (site_matches["placement_date"] <= date)
        & (site_matches["last_move_date"].fillna(pd.Timestamp.max) >= date)
    ]
    if not active.empty:
        split = cost / len(active)
        for _, c in active.iterrows():
            rows.append({
                "cycle_id": c["cycle_id"], "date": date,
                "cost_category": acct["Cost_Category"],
                "cost_subcategory": acct["Cost_Subcategory"],
                "allocated_cost": split, "confidence": "MEDIUM",
                "vendor": acct["Vendor"],
            })
        continue

    # LOW — Site only fallback
    split = cost / len(site_matches)
    for _, c in site_matches.iterrows():
        rows.append({
            "cycle_id": c["cycle_id"], "date": date,
            "cost_category": acct["Cost_Category"],
            "cost_subcategory": acct["Cost_Subcategory"],
            "allocated_cost": split, "confidence": "LOW",
            "vendor": acct["Vendor"],
        })

cost_cols = ["cycle_id", "date", "cost_category", "cost_subcategory",
             "allocated_cost", "confidence", "vendor"]
cycle_costs = pd.DataFrame(rows, columns=cost_cols)

print(f"Allocated {len(cycle_costs)} cost rows")
if not cycle_costs.empty:
    print("Confidence breakdown:", cycle_costs["confidence"].value_counts().to_dict())
cycle_costs.head()

Allocated 0 cost rows


,cycle_id,date,cost_category,cost_subcategory,allocated_cost,confidence,vendor


## 6. Hedge P&L

Without external futures files, settlement price is approximated as the average packer `Eval_Price_CWT` in the month of contract expiration. For SELL positions:

    gain = (strike - settlement) * head_covered * carcass_weight / 100

In [38]:
p = packer.dropna(subset=["Eval_Price_CWT", "Kill_Date"]).copy()
p["yyyymm"] = p["Kill_Date"].dt.to_period("M")
monthly_eval = p.groupby("yyyymm")["Eval_Price_CWT"].mean()

hedge_rows = []
for _, h in hedging.iterrows():
    exp = h["Expiration_Date"]
    if pd.isna(exp):
        settle = np.nan
    else:
        ym = pd.Timestamp(exp).to_period("M")
        settle = monthly_eval.get(ym, np.nan)

    sign = 1 if str(h["Buy_Sell"]).upper() == "SELL" else -1
    if pd.isna(settle):
        gain = 0.0
    else:
        gain = sign * (h["Strike_or_Futures_Price_CWT"] - settle) \
               * h["Head_Covered"] * ASSUMED_CARC_WT_LB / 100.0
    hedge_rows.append({
        "cycle_id": h["Pig_Group_ID"],
        "head_covered": h["Head_Covered"],
        "strike_cwt": h["Strike_or_Futures_Price_CWT"],
        "settlement_cwt": settle,
        "hedge_gain_loss": round(gain, 2),
    })

hedge_pnl = pd.DataFrame(hedge_rows)
print(f"Computed {len(hedge_pnl)} hedge records")
hedge_pnl

Computed 18 hedge records


,cycle_id,head_covered,strike_cwt,settlement_cwt,hedge_gain_loss
0,PG-1017,2000,79.93,88.792222,-38107.56
1,PG-1014,1600,93.15,88.792222,14990.76
2,PG-1002,1200,88.20,88.792222,1527.93
3,PG-1015,1600,79.17,89.327500,34941.80
4,PG-1000,1200,89.89,88.152273,-4483.34
5,PG-1016,2000,77.18,89.327500,52234.25
6,PG-1011,2000,78.28,89.327500,47504.25
7,PG-1004,1600,75.18,87.830667,43518.29
8,PG-1005,1200,88.20,87.830667,-952.88
9,PG-1009,2000,93.23,87.830667,-23217.13


## 7. Per-cycle P&L

    net_pnl = packer_revenue + hedge_pnl - total_cost

Packer revenue is allocated by Site share of head.

In [39]:
# Revenue allocation by Site share
packer_by_site = (
    packer.groupby("Site")
    .agg(site_net_payment=("Net_Payment_$", "sum"),
         site_delivered_head=("Delivered_Head", "sum"))
    .reset_index()
    .rename(columns={"Site": "nursery_site"})
)
head_by_site = (
    cycles.groupby("nursery_site")["received_head"].sum()
    .reset_index().rename(columns={"received_head": "site_total_head"})
)

pnl = (
    cycles.merge(packer_by_site, on="nursery_site", how="left")
          .merge(head_by_site, on="nursery_site", how="left")
)
pnl["revenue_share"] = pnl["received_head"] / pnl["site_total_head"]
pnl["packer_revenue"] = (
    pnl["site_net_payment"].fillna(0) * pnl["revenue_share"].fillna(0)
)

# Aggregate costs and hedge P&L per cycle
cost_agg = (cycle_costs.groupby("cycle_id")["allocated_cost"].sum()
            .reset_index().rename(columns={"allocated_cost": "total_cost"}))
hedge_agg = (hedge_pnl.groupby("cycle_id")["hedge_gain_loss"].sum()
             .reset_index().rename(columns={"hedge_gain_loss": "hedge_pnl"}))

cycle_pnl = (
    pnl[["cycle_id", "received_head", "packer_revenue"]]
    .merge(cost_agg, on="cycle_id", how="left")
    .merge(hedge_agg, on="cycle_id", how="left")
)
cycle_pnl["total_cost"] = cycle_pnl["total_cost"].fillna(0)
cycle_pnl["hedge_pnl"] = cycle_pnl["hedge_pnl"].fillna(0)
cycle_pnl["net_pnl"] = (
    cycle_pnl["packer_revenue"] + cycle_pnl["hedge_pnl"] - cycle_pnl["total_cost"]
)
cycle_pnl["pnl_per_head"] = cycle_pnl["net_pnl"] / cycle_pnl["received_head"].replace(0, np.nan)
cycle_pnl = cycle_pnl.round(2)
cycle_pnl

,cycle_id,received_head,packer_revenue,total_cost,hedge_pnl,net_pnl,pnl_per_head
0,PG-1017,2285,0.0,0,-38107.56,-38107.56,-16.677269
1,PG-1002,2274,0.0,0,1527.93,1527.93,0.671913
2,PG-1014,2431,0.0,0,14990.76,14990.76,6.166499
3,PG-1011,2350,0.0,0,47504.25,47504.25,20.214574
4,PG-1015,2569,0.0,0,34941.80,34941.8,13.601323
5,PG-1016,2399,0.0,0,52234.25,52234.25,21.773343
6,PG-1000,2248,0.0,0,-4483.34,-4483.34,-1.994368
7,PG-1004,2106,0.0,0,43518.29,43518.29,20.663955
8,PG-1005,2107,0.0,0,-952.88,-952.88,-0.452245
9,PG-1009,2498,0.0,0,-23217.13,-23217.13,-9.294287


## 8. Anomaly flags

Rules-based outlier detection:
- **Z-score** outliers on `pnl_per_head` (|z| ≥ 2.5)
- **Hard rule**: any cycle with negative net P&L

In [40]:
flags = []
if cycle_pnl["pnl_per_head"].std(skipna=True) and cycle_pnl["pnl_per_head"].std() > 0:
    mean = cycle_pnl["pnl_per_head"].mean()
    std = cycle_pnl["pnl_per_head"].std()
    for _, r in cycle_pnl.iterrows():
        if pd.isna(r["pnl_per_head"]):
            continue
        z = (r["pnl_per_head"] - mean) / std
        if abs(z) >= Z_THRESHOLD:
            flags.append({
                "cycle_id": r["cycle_id"], "metric": "pnl_per_head",
                "value": r["pnl_per_head"], "z_score": round(z, 2),
                "severity": "HIGH" if abs(z) >= 3 else "MEDIUM",
                "note": "Statistical outlier vs other cycles",
            })

for _, r in cycle_pnl.iterrows():
    if r["net_pnl"] < 0:
        flags.append({
            "cycle_id": r["cycle_id"], "metric": "net_pnl",
            "value": r["net_pnl"], "z_score": None, "severity": "HIGH",
            "note": "Cycle finished with a loss",
        })

anomalies = pd.DataFrame(
    flags, columns=["cycle_id", "metric", "value", "z_score", "severity", "note"]
)
print(f"Flagged {len(anomalies)} issues")
anomalies

Flagged 5 issues


,cycle_id,metric,value,z_score,severity,note
0,PG-1017,net_pnl,-38107.56,None,HIGH,Cycle finished with a loss
1,PG-1000,net_pnl,-4483.34,None,HIGH,Cycle finished with a loss
2,PG-1005,net_pnl,-952.88,None,HIGH,Cycle finished with a loss
3,PG-1009,net_pnl,-23217.13,None,HIGH,Cycle finished with a loss
4,PG-1003,net_pnl,-21944.07,None,HIGH,Cycle finished with a loss


## 9. Save outputs + headline summary

In [41]:
cycles.to_csv(OUTPUT_DIR / "dim_cycle.csv", index=False)
cycle_costs.to_csv(OUTPUT_DIR / "fact_cycle_costs.csv", index=False)
hedge_pnl.to_csv(OUTPUT_DIR / "fact_hedge_pnl.csv", index=False)
cycle_pnl.to_csv(OUTPUT_DIR / "fact_cycle_pnl.csv", index=False)
anomalies.to_csv(OUTPUT_DIR / "fact_anomalies.csv", index=False)
dq_report.to_csv(OUTPUT_DIR / "dq_report.csv", index=False)

print("=" * 70)
print(" HEADLINE NUMBERS")
print("=" * 70)
print(f"  Cycles tracked:    {len(cycle_pnl)}")
print(f"  Total net P&L:     ${cycle_pnl['net_pnl'].sum():>14,.2f}")
print(f"  Avg P&L per head:  ${cycle_pnl['pnl_per_head'].mean():>14,.2f}")
print(f"  Cycles in the red: {(cycle_pnl['net_pnl'] < 0).sum()} / {len(cycle_pnl)}")
if not cycle_pnl.empty:
    best = cycle_pnl.loc[cycle_pnl['net_pnl'].idxmax()]
    worst = cycle_pnl.loc[cycle_pnl['net_pnl'].idxmin()]
    print(f"  Best cycle:        {best['cycle_id']} (${best['net_pnl']:,.2f})")
    print(f"  Worst cycle:       {worst['cycle_id']} (${worst['net_pnl']:,.2f})")

print()
print(f"Outputs written to: {OUTPUT_DIR.resolve()}")
for f in sorted(OUTPUT_DIR.glob("*.csv")):
    print(f"  - {f.name}")

 HEADLINE NUMBERS
  Cycles tracked:    18
  Total net P&L:     $    213,159.99
  Avg P&L per head:  $          4.82
  Cycles in the red: 5 / 18
  Best cycle:        PG-1016 ($52,234.25)
  Worst cycle:       PG-1017 ($-38,107.56)

Outputs written to: /workspaces/CCDS/Outputs
  - dim_cycle.csv
  - dq_report.csv
  - fact_anomalies.csv
  - fact_cycle_costs.csv
  - fact_cycle_pnl.csv
  - fact_hedge_pnl.csv
